In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import sqlite3
import matplotlib.pyplot as plt
import math
from collections import defaultdict
from dotenv import load_dotenv
from google import genai
from pandas.tseries.offsets import DateOffset
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import fbeta_score, make_scorer, average_precision_score, jaccard_score
from sklearn.model_selection import GridSearchCV, TunedThresholdClassifierCV
from itertools import product
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from pydantic import BaseModel
from typing import Literal, Optional
import json
from google.genai import types
import random
from google.genai import errors
import time
import os

In [ ]:
os.environ["LOKY_MAX_CPU_COUNT"] = "2"  # Replace 8 with your desired core count


load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

con = sqlite3.connect(database="data.sqlite")


data = (pd.read_sql_query(sql="SELECT * FROM cleaned_data",con=con,parse_dates={"Period"})
          .loc[:, lambda df: ~df.columns.isin(["Loan Identifier","Postal Code"])])
          
con.close()

In [3]:
target_col = "Default Next 12"
binary_cols = (data.columns[data.isin([0,1]).all(axis=0)]
                   .drop(target_col)
                   .to_numpy()
                   .tolist())
numeric_cols = (data.columns[data.apply(lambda x: pd.api.types.is_numeric_dtype(x),axis=0)]
                    .drop(target_col)
                    .drop(binary_cols)
                    .to_numpy()
                    .tolist())


# Binary Cols
# Choose only one from the lagged columns
print(f"Choosing binary lag columns")
groups = defaultdict(list)
for col in binary_cols:
    parts = col.rsplit(" ", 1)

    if len(parts) == 2 and parts[1].isnumeric():
        groups[parts[0]].append(col)

def binary_log_odds_sep(df,x_var,y_var):
    log_odds = (df.groupby(x_var).agg(default_sum =(y_var,"sum"),
                                      default_count = (y_var,"count"))
                  .reset_index())
    
    log_odds["log odds ratio"] = log_odds.apply(lambda x: math.log(x["default_sum"]/x["default_count"]) - 
                                                          math.log(1 - (x["default_sum"]/x["default_count"])), axis=1)
    
    return abs(log_odds.loc[(log_odds[x_var]==1),"log odds ratio"].values[0] - 
               log_odds.loc[(log_odds[x_var]==0),"log odds ratio"].values[0])

    

for group, col_list in groups.items():
    max_sep = 0
    max_sep_index = 0
    df = data.loc[data["Period"]<"2012-01-01",col_list + [target_col]]
    for i in range(len(col_list)):
        sep = binary_log_odds_sep(df,col_list[i],target_col)
        if sep > max_sep:
            max_sep = sep
            max_sep_index = i

    print(f"Best lag column for {group} is {col_list[max_sep_index]}")
    for col in col_list:
        if col != col_list[max_sep_index]:
            binary_cols.remove(col)
            print(f"Removing {col}")

# Remove columns with high Jaccard similarity
removed_cols = []

for i in binary_cols:
    if i in removed_cols:
        continue
    for j in binary_cols:
        if j in removed_cols or i==j:
            continue
        jaccard = jaccard_score(data.loc[data["Period"]<"2012-01-01",i],data.loc[data["Period"]<"2012-01-01",j])
        if jaccard >= 0.8:
            print(f"Jaccard similarity between {i} and {j} is {jaccard}")
            print(f"Removing {j}")
            removed_cols.append(j)

if removed_cols == []:
    print("No binary columns removed due to high Jaccard similarity")
else:
    for col in removed_cols:
        binary_cols.remove(col)


Choosing binary lag columns
Best lag column for Moderate Disaster is Moderate Disaster 6
Removing Moderate Disaster 1
Removing Moderate Disaster 3
Removing Moderate Disaster 9
Best lag column for Extreme Disaster is Extreme Disaster 3
Removing Extreme Disaster 1
Removing Extreme Disaster 6
Removing Extreme Disaster 9
No binary columns removed due to high Jaccard similarity


In [4]:
# Numeric Columns
# For macroeconomic lags, we choose the one with the most unique information
print(f"Choosing numeric lag columns")
groups = defaultdict(list)
for col in numeric_cols:
    parts = col.rsplit(" ", 1)

    if len(parts) == 2 and parts[1].isnumeric():
        groups[parts[0]].append(col)




for group, col_list in groups.items():
    
    # We look at max correlation with rest of independent variables
    # and choose the one with the lowest max correlation
    min_max_corr = 1
    min_max_corr_index = 0
    other_cols = list(set(numeric_cols) - set(col_list))
    df =  data.loc[data["Period"]<"2012-01-01",:]
    for i in range(len(col_list)):
        max_corr = 0
        for other_col in other_cols:
            corr = np.corrcoef(df[col_list[i]],df[other_col])[0,1]
            if corr > max_corr:
                max_corr = corr
        if max_corr < min_max_corr:
            min_max_corr = max_corr
            min_max_corr_index = i
        

    print(f"Best lag column for {group} is {col_list[min_max_corr_index]}")
    for col in col_list:
        if col != col_list[min_max_corr_index]:
            numeric_cols.remove(col)
            print(f"Removing {col}")

# Now remove columns with multicollinearity
removed_cols = []

for i in numeric_cols:
    if i in removed_cols:
        continue
    for j in numeric_cols:
        if j in removed_cols or i==j:
            continue
        corr = np.corrcoef(data.loc[data["Period"]<"2012-01-01",i],data.loc[data["Period"]<"2012-01-01",j])[0,1]
        if corr >= 0.8:
            print(f"Correlation between {i} and {j} is {corr}")
            print(f"Removing {j}")
            removed_cols.append(j)

if removed_cols == []:
    print("No numeric columns removed due to multicollinearity")
else:
    for col in removed_cols:
        numeric_cols.remove(col)


Choosing numeric lag columns
Best lag column for FEDFUNDS is FEDFUNDS 9
Removing FEDFUNDS 3
Removing FEDFUNDS 6
Best lag column for GS5 is GS5 3
Removing GS5 6
Removing GS5 9
Best lag column for UNRATE is UNRATE 9
Removing UNRATE 3
Removing UNRATE 6
Best lag column for INDPRO change is INDPRO change 3
Removing INDPRO change 6
Removing INDPRO change 9
Best lag column for CPI change is CPI change 9
Removing CPI change 3
Removing CPI change 6
Correlation between GS5 3 and FEDFUNDS 9 is 0.8647794002923891
Removing FEDFUNDS 9


In [5]:
def bin_table(df,x_var):
    df = df[[x_var,"Default Next 12"]].copy()
    df["bin"] = pd.cut(df[x_var],50)
    df["left_edge"] = df["bin"].apply(lambda x:str(x).split(", ")[0].replace("(","")).astype(float)
    df["right_edge"] = df["bin"].apply(lambda x:str(x).split(", ")[1].replace("]","")).astype(float)
    
    log_odds = (df.groupby("bin",observed=False).agg(n_defaults =("Default Next 12","sum"),
                                                 n_obs = ("Default Next 12","count"),
                                                 left_bin_edge = ("left_edge","mean"),
                                                 right_bin_edge = ("right_edge","mean"))
                  .reset_index())

    log_odds["bin_midpoint"] = (log_odds["left_bin_edge"] + log_odds["right_bin_edge"])/2
    log_odds["log_odds"] = np.log(log_odds["n_defaults"] + 0.5) - np.log(log_odds["n_obs"] - log_odds["n_defaults"] + 0.5)

    return log_odds

In [ ]:
class TransformationParameters(BaseModel):
    power: Optional[float] = None
    constant: Optional[float] = None
    knot: Optional[float] = None

class TransformationRecommendation(BaseModel):
    variable_name: str

    # Observed relationship
    linearity_assessment: Literal[
        "Linear",
        "Nonlinear",
        "Unclear"
    ]

    relationship_pattern: Literal[
        "Increasing",
        "Decreasing",
        "U-shaped",
        "Inverted U-shaped",
        "Irregular",
        "No clear relationship",
        "Unclear"
    ]

    observed_pattern: str

    # Recommended transformation
    recommended_transformation: Literal[
        "identity",
        "log",
        "sqrt",
        "power",
        "reciprocal",
        "hinge"
    ]

    generated_columns: list[str]

    parameters: TransformationParameters

    joint_mapping_injective: bool

    reasoning: str

    confidence: Literal["Low", "Medium", "High"]

    additional_checks: list[str]

In [ ]:
def analyze_linearity(variable, bin_table):

   # Convert pandas table to JSON
   bin_data = json.loads(bin_table.to_json(orient="records"))

   prompt = f"""
   You are an expert credit risk modeller specializing
   in mortgage probability of default modelling.

   Analyze the following binned log-odds data for the variable {variable}:

   {bin_data}

   Choose the most appropriate transformation from this predefined library:

   1. identity:
      Keep the original variable unchanged.

   2. log:
      Generate log1p(x).

   3. sqrt:
      Generate sqrt(x).

   4. power:
      Generate x raised to one of these powers: 0.5, 1.5, or 2.

   5. reciprocal:
      Generate 1 / (x + c), where c must be a positive constant.

   6. hinge:
      Generate two columns:
      min(x, k) and max(x - k, 0).

   IMPORTANT CONSTRAINTS:

   - Only recommend transformations from this library.
   - Prefer simple transformations implementable with standard pandas/numpy operations.
   - If multiple columns are generated, their combined mapping must be injective:
   distinct original values must produce distinct feature vectors.
   - Do not recommend arbitrary polynomials, unrestricted splines,
   custom functions, or specialized libraries.
   - Recommend identity if the evidence does not justify a transformation.
   - Do not assume a relationship is nonlinear merely because the bins
   fluctuate. Consider bin sample sizes and instability.
   - Do not invent numerical patterns or knot locations unsupported
   by the supplied data.
   - Treat confidence as a qualitative assessment, not a statistical probability.

   Return ONLY valid JSON with this structure:

   {{
      "variable_name": {variable},
      "recommended_transformation": "identity | log | sqrt | power | reciprocal | hinge",
      "parameters": {{
         "power": null,
         "constant": null,
         "knot": null
      }},
      "generated_columns": ["column_name_1", "column_name_2"],
      "joint_mapping_injective": true,
      "reasoning": "Explanation based on the supplied binned data",
      "confidence": "Low | Medium | High",
      "additional_checks": []
   }}

   For parameters that are not relevant, return null.

   In addition to recommending a transformation, assess the observed
   relationship between the predictor and the log-odds of default.

   Return:

   - linearity_assessment:
   Whether the relationship appears linear, nonlinear, or unclear.

   - relationship_pattern:
   The overall observed shape of the relationship.

   - observed_pattern:
   A detailed description of how the log-odds change across the
   predictor's range. Mention relevant turning points, ranges,
   and potential irregularities.

   Base your assessment strictly on the supplied binned data.
   Do not interpret fluctuations in bins with small sample sizes
   as reliable patterns. Distinguish observed patterns from
   possible explanations for those patterns.
   """
   
   max_retries = 10
   for attempt in range(max_retries):

      try:
         response = client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=prompt,
            config=types.GenerateContentConfig(
               response_mime_type="application/json",
               response_schema=TransformationRecommendation
            )
         )

         return response.parsed

      except errors.ServerError as e:

         if e.code != 503 or attempt == max_retries - 1:
            raise

         wait = min(4 ** attempt, 30) + random.uniform(0, 1)

         print(f"Gemini unavailable. Retrying in {wait:.1f}s...")
         time.sleep(wait)

In [8]:
def apply_transformation(df, recommendation, numeric_cols):

    variable = recommendation.variable_name
    transformation = recommendation.recommended_transformation
    params = recommendation.parameters
    columns = recommendation.generated_columns

    x = df[variable]

    if not recommendation.joint_mapping_injective:
        raise ValueError("Transformation does not preserve the original variable's information.")

    if transformation == "identity":
        return df

    elif transformation == "log":
        if (x < 0).any():
            raise ValueError("Log1p transformation requires values >= 0.")

        df[columns[0]] = np.log1p(x)
        numeric_cols.append(columns[0])
        numeric_cols.remove(variable)

    elif transformation == "sqrt":
        if (x < 0).any():
            raise ValueError("Square root transformation requires values >= 0.")

        df[columns[0]] = np.sqrt(x)
        numeric_cols.append(columns[0])
        numeric_cols.remove(variable)

    elif transformation == "power":
        power = params.power

        if power is None:
            raise ValueError("Missing power parameter.")

        df[columns[0]] = x ** power
        numeric_cols.append(columns[0])
        numeric_cols.remove(variable)

    elif transformation == "reciprocal":
        constant = params.constant

        if constant is None or (x + constant <= 0).any():
            raise ValueError("Invalid reciprocal constant.")

        df[columns[0]] = 1 / (x + constant)
        numeric_cols.append(columns[0])
        numeric_cols.remove(variable)

    elif transformation == "hinge":
        knot = params.knot

        if knot is None or len(columns) != 2:
            raise ValueError("Invalid hinge parameters.")

        df[columns[0]] = x.clip(upper=knot)
        df[columns[1]] = (x - knot).clip(lower=0)
        numeric_cols.append(columns[0])
        numeric_cols.append(columns[1])
        numeric_cols.remove(variable)


    else:
        raise ValueError(f"Unsupported transformation: {transformation}")

    return

In [9]:
numeric_cols_copy = numeric_cols.copy()
for col in numeric_cols_copy:
    log_odds = bin_table(data.loc[data["Period"]<"2012-01-01",:],col)
    result = analyze_linearity(variable=col,bin_table=log_odds)
    print(f"Linearity assessment for {col}: {result.linearity_assessment}, {result.relationship_pattern}")
    print(f"{result.observed_pattern}")
    print(f"Apply {result.recommended_transformation} transformation")
    apply_transformation(data,result,numeric_cols)

Linearity assessment for Classic FICO®: Linear, Decreasing
The log-odds of default generally decrease as the Classic FICO score increases, moving from approximately -2.5 at lower valid scores (around 478) down to around -5.5 to -6.1 at higher scores (around 780 to 800), reflecting the expected monotonic decrease in default risk for higher credit scores.
Apply identity transformation
Linearity assessment for Original Interest Rate: Linear, No clear relationship
The log-odds fluctuate across the bins without a clear monotonic or non-linear trend, especially showing high variance in bins with very low sample sizes.
Apply identity transformation
Linearity assessment for Original Debt-to-Income (DTI) Ratio: Nonlinear, U-shaped
The log-odds of default show a U-shaped or curved pattern across the DTI range. Initially, at very low DTI levels (e.g., below 10), log-odds are relatively high but volatile due to small sample sizes. As DTI increases into the mid-range (10 to 30), log-odds stabilize 

In [10]:
data = data.loc[:,numeric_cols + binary_cols + [target_col,"Period"]]

In [ ]:
# Hyperparameter and threshold tuning
# We don't have enough variables to need to worry about penalization, but gradient boosting has hyperparameters
# To mimic how we test model performance, we will have a 3 year training window with a 1 year gap between training and test

# Construct a hyperparameter grid
params = {"max_iter":[100,200,300],"learning_rate":[0.03,0.05,0.1],"max_depth":[None,10]}
keys = params.keys()
combinations = product(*params.values())

# Convert to a list of dictionaries
grid = [dict(zip(keys, values)) for values in combinations]

cv_results = []

for param_set in grid:
    learning_rate = param_set["learning_rate"]
    max_iter = param_set["max_iter"]
    max_depth = param_set["max_depth"]
    yearly_scores = []
    curr_period = pd.Timestamp("2001-12-01")
    while curr_period <= pd.Timestamp("2009-12-01"):
        X_train = data.loc[(data["Period"] > curr_period - DateOffset(years=3)) & (data["Period"] <= curr_period),numeric_cols + binary_cols]
        y_train = data.loc[(data["Period"] > curr_period - DateOffset(years=3)) & (data["Period"] <= curr_period),target_col]

        X_test = data.loc[(data["Period"] > curr_period + DateOffset(years=1)) &
                        (data["Period"] <= curr_period + DateOffset(years=2)),numeric_cols + binary_cols]
        y_test = data.loc[(data["Period"] > curr_period + DateOffset(years=1)) &
                        (data["Period"] <= curr_period + DateOffset(years=2)),target_col]

        est_gb = (HistGradientBoostingClassifier(random_state=0,learning_rate=learning_rate,max_iter=max_iter,max_depth=max_depth)
                  .fit(X_train,y_train))

        y_pred_prob = est_gb.predict_proba(X_test)[:,1]
        score = average_precision_score(y_test,y_pred_prob)
        yearly_scores.append(score)
        curr_period = curr_period + DateOffset(years=1)

    cv_results.append(sum(yearly_scores) / len(yearly_scores))

best_params = grid[np.argmax(cv_results)]
print(f"Best hyperparameters for gradient boosting: {best_params}")


Best hyperparameters for gradient boosting: {'max_iter': 100, 'learning_rate': 0.03, 'max_depth': 10}


In [14]:
# Decision threshold tuning will deal with our class imbalance problem
# As missing defaults is more costly than misclassifying defaults (i.e. providing assistance to those who don't need it),
# we use an f2 score
threshold_grid = np.arange(0.001,0.03,0.002)
learning_rate = best_params["learning_rate"]
max_iter = best_params["max_iter"]
max_depth = best_params["max_depth"]
cv_results_gb = []
cv_results_lr = []
est_gb = HistGradientBoostingClassifier(random_state=0,learning_rate=learning_rate,max_iter=max_iter,max_depth=max_depth)
est_lr = LogisticRegression(C=np.inf)
for threshold in threshold_grid:
    yearly_scores_gb = []
    yearly_scores_lr = []
    curr_period = pd.Timestamp("2001-12-01")
    while curr_period <= pd.Timestamp("2009-12-01"):
        scaler = StandardScaler()
        X_train = data.loc[(data["Period"] > curr_period - DateOffset(years=3)) & (data["Period"] <= curr_period),numeric_cols + binary_cols]
        y_train = data.loc[(data["Period"] > curr_period - DateOffset(years=3)) & (data["Period"] <= curr_period),target_col]

        X_test = data.loc[(data["Period"] > curr_period + DateOffset(years=1)) &
                        (data["Period"] <= curr_period + DateOffset(years=2)),numeric_cols + binary_cols]
        y_test = data.loc[(data["Period"] > curr_period + DateOffset(years=1)) &
                        (data["Period"] <= curr_period + DateOffset(years=2)),target_col]

        X_train[numeric_cols] = X_train[numeric_cols].astype(float)
        X_test[numeric_cols] = X_test[numeric_cols].astype(float)
        X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
        X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

        y_pred_gb = (est_gb.fit(X_train,y_train).predict_proba(X_test)[:,1] > threshold)
        y_pred_lr = (est_lr.fit(X_train,y_train).predict_proba(X_test)[:,1] > threshold)
        score_gb = fbeta_score(y_test, y_pred_gb, beta=2)
        score_lr = fbeta_score(y_test, y_pred_lr, beta=2)
        yearly_scores_gb.append(score_gb)
        yearly_scores_lr.append(score_lr)
        curr_period = curr_period + DateOffset(years=1)
    
    cv_results_gb.append(sum(yearly_scores_gb) / len(yearly_scores_gb))
    cv_results_lr.append(sum(yearly_scores_lr) / len(yearly_scores_lr))


best_threshold_gb = threshold_grid[np.argmax(cv_results_gb)]
print(f"Best decision threshold for gradient boosting: {best_threshold_gb}")
best_threshold_lr = threshold_grid[np.argmax(cv_results_lr)]
print(f"Best decision threshold for logistic regression: {best_threshold_lr}")





Best decision threshold for gradient boosting: 0.009000000000000001
Best decision threshold for logistic regression: 0.019000000000000003


In [15]:


con = sqlite3.connect(database="data.sqlite")
(data
  .to_sql(name="final_data", 
          con=con, 
          if_exists="replace",
          index=False)
)
con.close()